In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output


def make_figure(n_tosses=100, p_heads=0.5, seed=42):
    rng = np.random.default_rng(seed)

    tosses = rng.binomial(1, p_heads, size=n_tosses)

    df = pd.DataFrame({
        "Toss": np.arange(1, n_tosses + 1),
        "Head": tosses
    })

    df["Tail"] = 1 - df["Head"]
    df["Cumulative_Heads"] = df["Head"].cumsum()
    df["Cumulative_Tails"] = df["Tail"].cumsum()
    df["Running_Proportion_Heads"] = df["Cumulative_Heads"] / df["Toss"]

    n_heads = int(df["Head"].sum())
    n_tails = int(df["Tail"].sum())
    observed_p_heads = n_heads / n_tosses

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Sequence of Toss Outcomes",
            "Running Proportion of Heads",
            "Final Counts",
            "Summary"
        ),
        specs=[
            [{"type": "scatter"}, {"type": "scatter"}],
            [{"type": "bar"}, {"type": "table"}]
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.10
    )

    fig.add_trace(
        go.Scatter(
            x=df["Toss"],
            y=df["Head"],
            mode="markers",
            marker=dict(size=7),
            hovertemplate="Toss %{x}<br>Outcome %{y}<extra></extra>"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df["Toss"],
            y=df["Running_Proportion_Heads"],
            mode="lines",
            hovertemplate="Toss %{x}<br>Observed p̂ = %{y:.3f}<extra></extra>"
        ),
        row=1, col=2
    )

    fig.add_hline(
        y=p_heads,
        line_dash="dash",
        annotation_text=f"Theoretical P(Heads) = {p_heads:.2f}",
        row=1, col=2
    )

    fig.add_trace(
        go.Bar(
            x=["Heads", "Tails"],
            y=[n_heads, n_tails],
            hovertemplate="%{x}: %{y}<extra></extra>"
        ),
        row=2, col=1
    )

    fig.add_trace(
        go.Table(
            header=dict(values=["Metric", "Value"]),
            cells=dict(
                values=[
                    [
                        "Number of tosses",
                        "Theoretical P(Heads)",
                        "Observed Heads",
                        "Observed Tails",
                        "Observed proportion of Heads",
                        "Difference: observed - theoretical"
                    ],
                    [
                        n_tosses,
                        f"{p_heads:.2f}",
                        n_heads,
                        n_tails,
                        f"{observed_p_heads:.3f}",
                        f"{observed_p_heads - p_heads:+.3f}"
                    ]
                ]
            )
        ),
        row=2, col=2
    )

    fig.update_yaxes(
        title_text="Outcome (1 = Heads, 0 = Tails)",
        tickmode="array",
        tickvals=[0, 1],
        ticktext=["Tails", "Heads"],
        row=1, col=1
    )
    fig.update_xaxes(title_text="Toss number", row=1, col=1)

    fig.update_yaxes(
        title_text="Proportion of Heads",
        range=[0, 1],
        row=1, col=2
    )
    fig.update_xaxes(title_text="Toss number", row=1, col=2)

    fig.update_yaxes(title_text="Count", row=2, col=1)

    fig.update_layout(
        title="Coin Toss Simulation Dashboard",
        height=750,
        width=1100,
        showlegend=False,
        template="plotly_white"
    )

    return fig


coin_toggle = widgets.ToggleButtons(
    options=["Fair coin", "Biased coin"],
    value="Fair coin",
    description="Coin type:",
    style={"description_width": "initial"}
)

coin_help = widgets.HTML(
    value="""
    <div style="font-size:14px; color:#444; margin:0 0 10px 110px;">
    <b>Coin type</b> chooses whether the coin is balanced or intentionally biased.<br>
    • <b>Fair coin</b>: Heads and Tails are equally likely, so P(Heads) = 0.50<br>
    • <b>Biased coin</b>: Heads and Tails are not equally likely
    </div>
    """
)

n_slider = widgets.IntSlider(
    value=100,
    min=10,
    max=1000,
    step=10,
    description="Tosses:",
    style={"description_width": "initial"},
    continuous_update=False
)

n_help = widgets.HTML(
    value="""
    <div style="font-size:14px; color:#444; margin:0 0 10px 110px;">
    <b>Tosses</b> is the total number of coin flips in the experiment.<br>
    Small numbers produce more random fluctuation.<br>
    Large numbers usually make the observed proportion move closer to the theoretical probability.
    </div>
    """
)

p_slider = widgets.FloatSlider(
    value=0.5,
    min=0.05,
    max=0.95,
    step=0.05,
    description="P(Heads):",
    readout_format=".2f",
    style={"description_width": "initial"},
    continuous_update=False
)

p_help = widgets.HTML(
    value="""
    <div style="font-size:14px; color:#444; margin:0 0 10px 110px;">
    <b>P(Heads)</b> is the theoretical probability that one toss results in Heads.<br>
    • 0.50 means a fair coin<br>
    • Above 0.50 means Heads is more likely<br>
    • Below 0.50 means Tails is more likely
    </div>
    """
)

seed_slider = widgets.IntSlider(
    value=42,
    min=1,
    max=500,
    step=1,
    description="Seed:",
    style={"description_width": "initial"},
    continuous_update=False
)

seed_help = widgets.HTML(
    value="""
    <div style="font-size:14px; color:#444; margin:0 0 10px 110px;">
    <b>Seed</b> controls the random number generator.<br>
    If you use the same seed, you get the same simulated results again.<br>
    Changing the seed gives a different random path while keeping the same underlying probability.
    </div>
    """
)

run_button = widgets.Button(
    description="Run simulation",
    button_style="success"
)

output = widgets.Output()


def sync_coin_type(change):
    if change["new"] == "Fair coin":
        p_slider.value = 0.5
        p_slider.disabled = True
    else:
        p_slider.disabled = False
        if p_slider.value == 0.5:
            p_slider.value = 0.7


coin_toggle.observe(sync_coin_type, names="value")
p_slider.disabled = True


def render_dashboard():
    with output:
        clear_output(wait=True)
        fig = make_figure(
            n_tosses=n_slider.value,
            p_heads=p_slider.value,
            seed=seed_slider.value
        )
        display(fig)


def on_run_clicked(b):
    render_dashboard()


run_button.on_click(on_run_clicked)

controls = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:8px;'>Simulation Settings</h3>"),
    coin_toggle,
    coin_help,
    n_slider,
    n_help,
    p_slider,
    p_help,
    seed_slider,
    seed_help,
    run_button
])

display(controls)
display(output)

render_dashboard()

Output()